In [10]:
import pandas as pd
import os
df = pd.read_csv('MementoML.csv', skipinitialspace=True)

##### Mean auc for each dataset for each model

In [11]:
summary = df.groupby(['dataset', 'model', 'param_index'])['auc'].mean().reset_index()

##### Calculating mean auc for each model and hyperparams across datasets

In [12]:
final_ranking = summary.groupby(['model', 'param_index'])['auc'].mean().reset_index()

In [13]:
final_ranking = final_ranking.rename(columns={'auc': 'avg_auc_across_datasets'})

In [14]:
final_ranking = final_ranking.sort_values(by='avg_auc_across_datasets', ascending=False)
print(final_ranking)

         model  param_index  avg_auc_across_datasets
7410   xgboost         1004                 0.999509
7476   xgboost         1070                 0.999296
2329  catboost         1319                 0.999276
2284  catboost         1274                 0.999269
2282  catboost         1272                 0.999258
...        ...          ...                      ...
995   agtboost          996                      NaN
996   agtboost          997                      NaN
997   agtboost          998                      NaN
998   agtboost          999                      NaN
999   agtboost         1000                      NaN

[7602 rows x 3 columns]


##### Top 2 hyperparameter sets for each model

In [15]:
top2_per_model = final_ranking.groupby('model').head(2)

print(top2_per_model)

             model  param_index  avg_auc_across_datasets
7410       xgboost         1004                 0.999509
7476       xgboost         1070                 0.999296
2329      catboost         1319                 0.999276
2284      catboost         1274                 0.999269
595       agtboost          596                 0.991340
611       agtboost          612                 0.990961
5422  randomForest         1049                 0.918112
5419  randomForest         1046                 0.918104
6587        ranger         1203                 0.916791
6590        ranger         1206                 0.916312
1000   bartMachine            1                 0.910870
2739           gbm         1399                 0.908749
2612           gbm         1272                 0.908711
1009   bartMachine           10                 0.905041
5316          kknn         1954                 0.882962
5314          kknn         1952                 0.882962
3522        glmnet         1171

In [21]:
unique_models = top2_per_model['model'].unique()

all_params_dfs = []

for model_name in unique_models:
    file_path = os.path.join('parameters', f'{model_name}_params.csv')

    if os.path.exists(file_path):
        params_df = pd.read_csv(file_path)
        params_df['model'] = model_name
        all_params_dfs.append(params_df)

if all_params_dfs:
    full_params_df = pd.concat(all_params_dfs, ignore_index=True)

    final_combined = pd.merge(
        top2_per_model, 
        full_params_df, 
        on=['model', 'param_index'], 
        how='left'
    )

    print("Połączone wyniki z parametrami:")
else:
    print("Nie wczytano żadnych plików parametrów.")

Połączone wyniki z parametrami:


In [22]:
final_combined

,model,param_index,avg_auc_across_datasets,booster,nrounds,eta,subsample,max_depth,min_child_weight,colsample_bytree,...,splitrule,n.trees,interaction.depth,n.minobsinnode,shrinkage,bag.fraction,k,distance,alpha,lambda
0,xgboost,1004,0.999509,gbtree,236.0,0.210900,0.817738,13.0,7.025267,0.709598,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,xgboost,1070,0.999296,gbtree,355.0,0.032411,0.867936,8.0,2.189216,0.710233,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,catboost,1319,0.999276,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,catboost,1274,0.999269,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,agtboost,596,0.991340,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,agtboost,612,0.990961,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,randomForest,1049,0.918112,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,randomForest,1046,0.918104,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,ranger,1203,0.916791,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,gini,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,ranger,1206,0.916312,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,gini,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Catboost params

In [26]:
catboost_results = top2_per_model[top2_per_model['model'] == 'catboost']
file_path = os.path.join('parameters', 'catboost_params.csv')

if os.path.exists(file_path):
    params_df = pd.read_csv(file_path)
    # Wybieramy tylko te dwa indeksy (1319 i 1274)
    detailed_params = params_df[params_df['param_index'].isin([1319, 1274])]
    
    # Łączymy z wynikiem AUC dla pełnego obrazu
    final_view = pd.merge(catboost_results, detailed_params, on='param_index')
    
    print("Najlepsze konfiguracje CatBoost z parametrami:")
    print(final_view)

Najlepsze konfiguracje CatBoost z parametrami:
      model  param_index  avg_auc_across_datasets  iterations  depth  \
0  catboost         1319                 0.999276         278      9   
1  catboost         1274                 0.999269        6455     10   

   l2_leaf_reg  bagging_temperature  learning_rate  
0     1.447357             1.602599       0.729353  
1     3.452452             2.403609       1.165797  


In [24]:
catboost_results

,model,param_index,avg_auc_across_datasets
2329,catboost,1319,0.999276
2284,catboost,1274,0.999269


#### GBM params

In [27]:
catboost_results = top2_per_model[top2_per_model['model'] == 'gbm']
file_path = os.path.join('parameters', 'gbm_params.csv')

if os.path.exists(file_path):
    params_df = pd.read_csv(file_path)
    detailed_params = params_df[params_df['param_index'].isin([1399, 1272])]

    final_view = pd.merge(catboost_results, detailed_params, on='param_index')
    
    print("Najlepsze konfiguracje GBM z parametrami:")
    print(final_view)

Najlepsze konfiguracje GBM z parametrami:
  model  param_index  avg_auc_across_datasets  n.trees  interaction.depth  \
0   gbm         1399                 0.908749     9751                  5   
1   gbm         1272                 0.908711     8039                  5   

   n.minobsinnode  shrinkage  bag.fraction  
0               3   0.003642      0.735056  
1               3   0.005549      0.814662  
